# Final application + historical data preparation

We add historical features to the application tables, keep the saved training/validation groups, and prepare numeric inputs for a Logistic Regression baseline. No prediction model is trained here.

**Inputs:** `applications_featured.joblib`, `application_test_featured.joblib`, `historical_features.parquet`, and the saved train/validation IDs.

The application tables have already been cleaned and feature-engineered, but have not been filled, encoded, or scaled. We do not start from the already-encoded application-only matrices.

**Two combined tables:** `application_train_with_history.parquet` includes TARGET; `application_test_with_history.parquet` does not. They are saved before learned filling/encoding so they remain useful for other models and cross-validation.

Run cells in order. This notebook writes new files under `combined/` folders only. Existing notebooks, raw CSVs, historical features, and application-only outputs are not overwritten.

## 1. Imports

`pandas` handles tables, `numpy` handles numeric values, `joblib` reads saved Python objects, and `numpy.save` saves prepared numeric arrays. The scikit-learn tools will fill missing values, encode text, and scale numbers.

In [1]:
from pathlib import Path
import json
import gc
import joblib
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

## 2. Project paths and new output folders

Find the project without a personal drive or username. `combined` keeps this version separate from our application-only baseline.

In [2]:
project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data/raw/home-credit-default-risk").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside the project folder.")

application_folder = project_root / "data/interim/application"
application_splits = project_root / "data/processed/application"
history_path = project_root / "data/processed/historical_features.parquet"
combined_folder = project_root / "data/interim/combined"
prepared_folder = project_root / "data/processed/combined"
preprocessor_folder = project_root / "artifacts/preprocessors/combined"
metadata_folder = project_root / "artifacts/metadata/combined"

for folder in (combined_folder, prepared_folder, preprocessor_folder, metadata_folder):
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", project_root)

Project: E:\Depi Notes Data Engineering\DEPI project R5\ProJect


## 3. Read and check historical features

There must be one row per `SK_ID_CURR`, unique column names, and no TARGET. Missing history is allowed. We do not guess that missing debt or delay means zero.

In [3]:
history = pd.read_parquet(history_path)

assert history["SK_ID_CURR"].notna().all()
assert history["SK_ID_CURR"].is_unique
assert history.columns.is_unique
assert "TARGET" not in history.columns

print("Historical table:", history.shape)
print("Historical features:", history.shape[1] - 1)
display(history.head(3).iloc[:, :8])

Historical table: (356255, 406)
Historical features: 405


   SK_ID_CURR  ...  bureau_credit_sum_mean
0      100002  ...           108131.945625
1      100003  ...           254350.125000
2      100004  ...            94518.900000

[3 rows x 8 columns]

## 4. Join history to application train

A left merge keeps every current application, including applications without history. `validate="one_to_one"` stops a join that would multiply rows. Keep TARGET in this combined table, but never use it as an input feature.

In [5]:
application_train = joblib.load(application_folder / "applications_featured.joblib")

assert application_train["SK_ID_CURR"].notna().all()
assert application_train["SK_ID_CURR"].is_unique
assert application_train.columns.is_unique
assert application_train["TARGET"].isin([0, 1]).all()

shared_columns = (set(application_train.columns) & set(history.columns)) - {"SK_ID_CURR"}
assert not shared_columns, f"Feature names overlap: {shared_columns}"

original_train_ids = application_train["SK_ID_CURR"].copy()
original_target = application_train["TARGET"].copy()

combined_train = application_train.merge(
    history, on="SK_ID_CURR", how="left", sort=False, validate="one_to_one"
)

assert combined_train["SK_ID_CURR"].tolist() == original_train_ids.tolist()
assert combined_train["TARGET"].tolist() == original_target.tolist()
assert combined_train.columns.is_unique
assert len(combined_train) == len(application_train)

combined_feature_columns = [column for column in combined_train.columns
                            if column not in ["SK_ID_CURR", "TARGET"]]
print("Combined train:", combined_train.shape)
display(combined_train.head(3).iloc[:, :8])

Combined train: (307511, 532)


   SK_ID_CURR  TARGET  ... CNT_CHILDREN AMT_INCOME_TOTAL
0      100002       1  ...            0         202500.0
1      100003       0  ...            0         270000.0
2      100004       0  ...            0          67500.0

[3 rows x 8 columns]

## 5. Inspect missing values and save the combined train table

This is an inspection, not a decision to remove every column above a missingness threshold. Actual selection and medians will use the training portion only.

Save before encoding. Then release large temporary tables from memory. `gc.collect()` asks Python to clean up unused memory.

In [6]:
train_missing_report = pd.DataFrame({
    "missing_count": combined_train.isna().sum(),
    "missing_percent": combined_train.isna().mean() * 100,
}).sort_values("missing_percent", ascending=False)
display(train_missing_report.head(15).round(2))

combined_train.to_parquet(combined_folder / "application_train_with_history.parquet", index=False)
train_row_count = len(combined_train)

del application_train, combined_train, original_target
gc.collect()

                                                  missing_count  missing_percent
credit_card_dpd_def_last_positive_month                  297522            96.75
credit_card_dpd_last_positive_month                      294933            95.91
pos_dpd_def_last_positive_month                          269350            87.59
credit_card_total_payment_to_minimum_mean                256205            83.32
pos_dpd_last_positive_month                              256070            83.27
credit_card_total_payment_to_balance_mean                255855            83.20
credit_card_current_payment_mean                         254669            82.82
credit_card_latest_current_payment_mean                  254669            82.82
credit_card_current_payment_max                          254669            82.82
installments_version0_late_record_rate                   254377            82.72
installments_version0_late_records                       254377            82.72
credit_card_latest_over100_u

## 6. Join history to application test

Test has no TARGET and must have the same input columns as train. Its application IDs must not overlap with the train IDs. We save a separate combined test table, not one train/test table.

In [7]:
application_test = joblib.load(application_folder / "application_test_featured.joblib")

assert "TARGET" not in application_test.columns
assert application_test["SK_ID_CURR"].notna().all()
assert application_test["SK_ID_CURR"].is_unique
assert application_test.columns.is_unique
assert not application_test["SK_ID_CURR"].isin(original_train_ids).any()
assert not ((set(application_test.columns) & set(history.columns)) - {"SK_ID_CURR"})

combined_test = application_test.merge(
    history, on="SK_ID_CURR", how="left", sort=False, validate="one_to_one"
)

assert combined_test["SK_ID_CURR"].tolist() == application_test["SK_ID_CURR"].tolist()
assert len(combined_test) == len(application_test)
assert combined_test.columns.is_unique
assert set(combined_test.columns) - {"SK_ID_CURR"} == set(combined_feature_columns)
assert "TARGET" not in combined_test.columns

# Put test features in exactly the same order as train features.
combined_test = combined_test[["SK_ID_CURR"] + combined_feature_columns]
combined_test.to_parquet(combined_folder / "application_test_with_history.parquet", index=False)
test_row_count = len(combined_test)
print("Combined test:", combined_test.shape)

# History is already saved in the two combined tables.
del application_test, combined_test, history, original_train_ids
gc.collect()

Combined test: (48744, 531)


## 7. Restore the original training and validation groups

Read the saved IDs. Application membership and row order stay the same as the application-only baseline. There is no new random split.

`.set_index()` lets us select applications by ID. `.loc[ids]` selects their rows in the saved order. `.pop("TARGET")` removes the answer from the input table and returns it separately.

We convert numeric features to compact `float32` before selecting groups to avoid large duplicate copies. ID and TARGET stay unchanged. The saved combined parquet tables keep their original precision.

In [8]:
train_labels = pd.read_csv(application_splits / "train_labels_ids.csv")
valid_labels = pd.read_csv(application_splits / "valid_labels_ids.csv")

assert train_labels["SK_ID_CURR"].is_unique
assert valid_labels["SK_ID_CURR"].is_unique
assert not train_labels["SK_ID_CURR"].isin(valid_labels["SK_ID_CURR"]).any()

combined = pd.read_parquet(combined_folder / "application_train_with_history.parquet")
assert set(train_labels["SK_ID_CURR"]) | set(valid_labels["SK_ID_CURR"]) == set(combined["SK_ID_CURR"])
# Compact numeric features before selecting large groups to limit memory.
for column in combined.select_dtypes(include="number").columns:
    if column not in ["SK_ID_CURR", "TARGET"]:
        combined[column] = combined[column].astype(np.float32)
combined = combined.set_index("SK_ID_CURR")

X_train = combined.loc[train_labels["SK_ID_CURR"]]
X_valid = combined.loc[valid_labels["SK_ID_CURR"]]
y_train = X_train.pop("TARGET")
y_valid = X_valid.pop("TARGET")

assert y_train.tolist() == train_labels["TARGET"].tolist()
assert y_valid.tolist() == valid_labels["TARGET"].tolist()
assert X_train.index.tolist() == train_labels["SK_ID_CURR"].tolist()
assert X_valid.index.tolist() == valid_labels["SK_ID_CURR"].tolist()
assert len(X_train) + len(X_valid) == train_row_count

print("Training:", X_train.shape, "Validation:", X_valid.shape)
del combined
gc.collect()

Training: (246008, 530) Validation: (61503, 530)


## 8. Select usable columns using training only

Remove columns that are entirely missing or constant in training. Keep partly observed columns; a high missing percentage alone does not prove uselessness. Apply the same list to validation and later test.

In [9]:
empty_columns = X_train.columns[X_train.isna().all()].tolist()
constant_columns = X_train.columns[X_train.nunique(dropna=False).eq(1)].tolist()
dropped_columns = sorted(set(empty_columns + constant_columns))
retained_columns = [column for column in X_train.columns if column not in dropped_columns]

X_train = X_train[retained_columns]
X_valid = X_valid[retained_columns]

assert not {"TARGET", "SK_ID_CURR"}.intersection(retained_columns)
print("Dropped columns:", dropped_columns)
print("Retained input columns:", len(retained_columns))

Dropped columns: []
Retained input columns: 530


## 9. Separate numeric and text inputs

Learn these column lists from training. Normalize text gaps to `np.nan`; actual labels such as `XNA` remain unchanged. Numeric inputs use `float32` to reduce memory; this introduces small rounding differences but does not fill missing values or estimate statistics. We check that conversion creates no infinities.

In [10]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()
assert len(numeric_columns) + len(categorical_columns) == len(retained_columns)

for column in categorical_columns:
    assert not X_train[column].eq("__MISSING__").any()
    for table in (X_train, X_valid):
        values = table[column].astype(object)
        table[column] = values.where(pd.notna(values), np.nan)

for table in (X_train, X_valid):
    for column in numeric_columns:
        if table[column].dtype != np.float32:
            table[column] = table[column].astype(np.float32)
    assert not np.isinf(table[numeric_columns].to_numpy()).any()

print("Numeric:", len(numeric_columns), "Text:", len(categorical_columns))
gc.collect()

Numeric: 514 Text: 16


## 10. Define the combined preprocessor

This is a new preprocessor for application + historical columns. The application-only preprocessor remains unchanged.

- Numeric: training median filling, then StandardScaler for the Logistic Regression baseline.
- Text: explicit `__MISSING__` label, then one-hot encoding.
- Unknown validation/test categories: ignore them without creating new columns.

These are standard scikit-learn objects. `Pipeline` runs steps in order; `ColumnTransformer` sends columns to the matching pipeline. Nothing is fitted in this cell. A tree-model experiment can use the saved unencoded combined tables to choose another preparation.

Most combined features are numeric and nonzero. Dense float32 output uses less storage in memory than a sparse matrix of this shape. We save `.npy` arrays, which can be opened simply with `np.load(path)`.

In [11]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
], remainder="drop", sparse_threshold=0.0)

## 11. Fit and prepare training only

`fit_transform` learns medians, means, standard deviations, and categories from the training portion only, then applies them. Validation and test do not participate in fitting. Check the result, save it, and release the large matrix before preparing the next group.

The labels CSV contains IDs and TARGET in exactly the same order as the matrix. Matrix columns are described by the manifest saved later.

In [12]:
X_train_prepared = preprocessor.fit_transform(X_train)
feature_names = preprocessor.get_feature_names_out()
X_train_prepared = X_train_prepared.astype(np.float32, copy=False)

assert X_train_prepared.shape == (len(y_train), len(feature_names))
assert np.isfinite(X_train_prepared).all()
train_matrix_shape = X_train_prepared.shape

# Compare a small source sample with the fitted transform before saving.
expected_sample = preprocessor.transform(X_train.head(5))
np.testing.assert_allclose(X_train_prepared[:5], expected_sample)

np.save(prepared_folder / "X_train.npy", X_train_prepared)
pd.DataFrame({"SK_ID_CURR": X_train.index.to_numpy(), "TARGET": y_train.to_numpy()}).to_csv(
    prepared_folder / "train_labels_ids.csv", index=False
)
joblib.dump(preprocessor, preprocessor_folder / "preprocessor.joblib", compress=3)

print("Prepared training:", train_matrix_shape)
del X_train_prepared, expected_sample, X_train, y_train
gc.collect()

Prepared training: (246008, 660)


## 12. Transform validation with the training rules

Use `transform`, never `fit_transform`. We do not create new medians or a new category layout. Preserve the saved validation IDs and targets.

In [13]:
X_valid_prepared = preprocessor.transform(X_valid).astype(np.float32, copy=False)
assert X_valid_prepared.shape == (len(y_valid), len(feature_names))
assert np.isfinite(X_valid_prepared).all()
valid_matrix_shape = X_valid_prepared.shape

np.save(prepared_folder / "X_valid.npy", X_valid_prepared)
pd.DataFrame({"SK_ID_CURR": X_valid.index.to_numpy(), "TARGET": y_valid.to_numpy()}).to_csv(
    prepared_folder / "valid_labels_ids.csv", index=False
)

print("Prepared validation:", valid_matrix_shape)
del X_valid_prepared, X_valid, y_valid
gc.collect()

Prepared validation: (61503, 660)


## 13. Transform test with the same training rules

Load the separate combined test table, select training's retained columns in their saved order, and apply the same preparation. Test has no TARGET and is not split.

In [14]:
combined_test = pd.read_parquet(combined_folder / "application_test_with_history.parquet")
test_ids = combined_test["SK_ID_CURR"].copy()
X_test = combined_test[retained_columns].copy()
del combined_test

for column in categorical_columns:
    values = X_test[column].astype(object)
    X_test[column] = values.where(pd.notna(values), np.nan)
for column in numeric_columns:
    X_test[column] = X_test[column].astype(np.float32)
assert not np.isinf(X_test[numeric_columns].to_numpy()).any()

X_test_prepared = preprocessor.transform(X_test).astype(np.float32, copy=False)
assert X_test_prepared.shape == (test_row_count, len(feature_names))
assert np.isfinite(X_test_prepared).all()
assert test_ids.is_unique
assert not test_ids.isin(train_labels["SK_ID_CURR"]).any()
assert not test_ids.isin(valid_labels["SK_ID_CURR"]).any()
test_matrix_shape = X_test_prepared.shape

np.save(prepared_folder / "X_test.npy", X_test_prepared)
test_ids.to_frame().to_csv(prepared_folder / "test_ids.csv", index=False)

print("Prepared test:", test_matrix_shape)
del X_test_prepared, X_test, test_ids
gc.collect()

Prepared test: (48744, 660)


## 14. Reload and verify the saved outputs

Verify each saved matrix, IDs, and labels. Also compare five saved rows with a fresh transform of their unencoded combined source. We load one matrix at a time to limit memory.

A successful check establishes consistent preparation, not prediction quality.

`mmap_mode="r"` lets NumPy read saved array values from disk without loading another entire array into memory.

In [15]:
loaded_preprocessor = joblib.load(preprocessor_folder / "preprocessor.joblib")
assert loaded_preprocessor.get_feature_names_out().tolist() == feature_names.tolist()

for name, expected_shape in [("train", train_matrix_shape), ("valid", valid_matrix_shape), ("test", test_matrix_shape)]:
    matrix = np.load(prepared_folder / f"X_{name}.npy", mmap_mode="r")
    labels_filename = "test_ids.csv" if name == "test" else f"{name}_labels_ids.csv"
    labels = pd.read_csv(prepared_folder / labels_filename)
    assert matrix.shape == expected_shape
    assert len(labels) == matrix.shape[0]
    assert labels["SK_ID_CURR"].is_unique
    assert np.isfinite(matrix).all()

    if name == "test":
        source_path = combined_folder / "application_test_with_history.parquet"
        assert "TARGET" not in labels.columns
    else:
        source_path = combined_folder / "application_train_with_history.parquet"
        expected_labels = train_labels if name == "train" else valid_labels
        assert labels["SK_ID_CURR"].tolist() == expected_labels["SK_ID_CURR"].tolist()
        assert labels["TARGET"].tolist() == expected_labels["TARGET"].tolist()

    # Filter the source by just five IDs rather than loading the full table again.
    sample_ids = labels["SK_ID_CURR"].head(5).tolist()
    sample = pd.read_parquet(source_path, filters=[("SK_ID_CURR", "in", sample_ids)])
    sample = sample.set_index("SK_ID_CURR").loc[sample_ids, retained_columns].copy()
    for column in categorical_columns:
        values = sample[column].astype(object)
        sample[column] = values.where(pd.notna(values), np.nan)
    sample[numeric_columns] = sample[numeric_columns].astype(np.float32)
    expected = loaded_preprocessor.transform(sample)
    np.testing.assert_allclose(matrix[:5], expected)
    print(name, matrix.shape, "reload and source comparison: PASS")
    del matrix, labels, sample, expected
    gc.collect()

train (246008, 660) reload and source comparison: PASS
valid (61503, 660) reload and source comparison: PASS
test (48744, 660) reload and source comparison: PASS


## 15. Save a short manifest and show the handoff

The manifest records column order, selected/dropped features, dimensions, package versions, and our decisions. It is a small description of the saved results, not another dataset.

For cross-validation later, fit preprocessing separately inside each training fold using the unencoded combined tables. Our current matrices are for the fixed saved training/validation split.

In [16]:
import importlib.metadata

manifest = {
    "scope": "application + historical features; no model trained",
    "split": "reuse IDs and order from application-only train_labels_ids.csv and valid_labels_ids.csv",
    "combined_train_rows": train_row_count,
    "combined_test_rows": test_row_count,
    "combined_input_features": len(combined_feature_columns),
    "retained_features": retained_columns,
    "dropped_columns": dropped_columns,
    "numeric_features": numeric_columns,
    "categorical_features": categorical_columns,
    "transformed_features": feature_names.tolist(),
    "matrix_shapes": {"train": train_matrix_shape, "valid": valid_matrix_shape, "test": test_matrix_shape},
    "decisions": [
        "Reuse existing application cleaning and engineered features; raw sources unchanged",
        "Left joins by SK_ID_CURR; preserve every application and its row order",
        "Keep missing historical measurements unknown until training-median imputation",
        "Drop only training-empty and training-constant columns",
        "Fit median filling, scaling and one-hot encoding on training portion only",
        "Use float32 numeric arrays; dense .npy avoids sparse overhead for these mostly nonzero historical features",
    ],
    "versions": {package: importlib.metadata.version(package) for package in ["pandas", "numpy", "scikit-learn", "joblib"]},
    "verification": "PASS: IDs, targets, shapes, finite values, saved reloads and five-row source comparisons",
}
(metadata_folder / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(pd.DataFrame([
    ["Training", train_matrix_shape[0], train_matrix_shape[1]],
    ["Validation", valid_matrix_shape[0], valid_matrix_shape[1]],
    ["Test", test_matrix_shape[0], test_matrix_shape[1]],
], columns=["Dataset", "Rows", "Prepared features"]))
print("Combined preparation complete. No prediction model trained.")

Combined preparation complete. No prediction model trained.


      Dataset    Rows  Prepared features
0    Training  246008                660
1  Validation   61503                660
2        Test   48744                660

## Output map and next step

| Folder | New files and purpose |
|---|---|
| `data/interim/combined/` | `application_train_with_history.parquet` and `application_test_with_history.parquet`: the two combined application-level tables, before encoding |
| `data/processed/combined/` | `X_train.npy`, `X_valid.npy`, `X_test.npy`, matching `train_labels_ids.csv`, `valid_labels_ids.csv`, and `test_ids.csv` |
| `artifacts/preprocessors/combined/` | `preprocessor.joblib`: fitted rules for the combined version |
| `artifacts/metadata/combined/` | `manifest.json`: feature order, dimensions, decisions, and checks |

**Next:** train and evaluate a model using the combined training and validation matrices. Compare it with the application-only baseline on the same validation applications. Test outcomes are hidden, so test cannot provide an evaluation score.

This notebook does not claim to re-audit every historical feature formula or prove historical availability at scoring time. It reuses the existing historical summaries and checks their joins and preparation.